# Backpropagation From Scratch

I've used PyTorch's autograd so many times without really thinking about what's happening underneath. This notebook is me actually working through it — no frameworks, just numpy and the chain rule.

The goal is simple: build a 2-layer network, train it on toy data, and verify my gradients are correct by comparing them to numerical estimates.

If the gradient check passes, I know I got it right.

## The Math

Network:
```
z1 = X · W1 + b1
a1 = ReLU(z1)
z2 = a1 · W2 + b2
a2 = softmax(z2)
L  = cross_entropy(a2, y)
```

Backward pass (chain rule applied layer by layer):
```
dL/dz2 = a2 - y          ← softmax + cross-entropy gradient simplifies to this
dL/dW2 = a1.T @ dL/dz2
dL/db2 = sum(dL/dz2)
dL/da1 = dL/dz2 @ W2.T
dL/dz1 = dL/da1 * (z1 > 0)   ← ReLU gradient
dL/dW1 = X.T @ dL/dz1
dL/db1 = sum(dL/dz1)
```

### Why does dL/dz2 = a2 - y?

Starting from cross-entropy loss:
```
L = -sum_j [ y_j * log(a_j) ]
```
and softmax:
```
a_j = exp(z_j) / sum_k exp(z_k)
```

Taking dL/dz_j requires the product rule through the softmax. Two cases:
- When i == j: da_i/dz_j = a_i(1 - a_i)
- When i != j: da_i/dz_j = -a_i * a_j

Plugging into the chain rule and simplifying (the cross terms collapse):
```
dL/dz_j = a_j - y_j
```

This is why softmax + cross-entropy are almost always used together — the gradient is clean.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Activation Functions

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def softmax(z):
    # subtract max per row for numerical stability
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, y_onehot):
    N = y_onehot.shape[0]
    # clip to avoid log(0)
    log_probs = np.log(probs.clip(min=1e-12))
    return -np.sum(y_onehot * log_probs) / N

## Weight Initialisation

Using He initialisation for ReLU layers — scales by sqrt(2/n_in). 
This keeps variance stable as we go deeper.

In [ ]:
def init_weights(input_size, hidden_size, output_size):
    W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
    b2 = np.zeros((1, output_size))
    return W1, b1, W2, b2

## Forward Pass

In [ ]:
def forward(X, W1, b1, W2, b2, y_onehot):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)
    loss = cross_entropy_loss(a2, y_onehot)

    cache = {'X': X, 'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2}
    return loss, cache

## Backward Pass

In [ ]:
def backward(cache, y_onehot, W2):
    N = y_onehot.shape[0]
    X, z1, a1, a2 = cache['X'], cache['z1'], cache['a1'], cache['a2']

    # output layer
    dz2 = (a2 - y_onehot) / N
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)

    # hidden layer
    da1 = dz2 @ W2.T
    dz1 = da1 * relu_grad(z1)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

## Gradient Check

Numerical approximation of the gradient:
```
dL/dθ ≈ (L(θ + ε) - L(θ - ε)) / (2ε)
```

If this matches our analytical gradient closely (relative error < 1e-5), backprop is correct.

In [ ]:
def numerical_gradient(X, y_onehot, W1, b1, W2, b2, param, eps=1e-5):
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = param[idx]

        param[idx] = orig + eps
        loss_plus, _ = forward(X, W1, b1, W2, b2, y_onehot)

        param[idx] = orig - eps
        loss_minus, _ = forward(X, W1, b1, W2, b2, y_onehot)

        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        param[idx] = orig
        it.iternext()
    return grad


def gradient_check(X, y_onehot, W1, b1, W2, b2):
    loss, cache = forward(X, W1, b1, W2, b2, y_onehot)
    grads = backward(cache, y_onehot, W2)

    for name, param, analytical in [
        ('W1', W1, grads['dW1']),
        ('W2', W2, grads['dW2']),
    ]:
        numerical = numerical_gradient(X, y_onehot, W1, b1, W2, b2, param)
        diff = np.linalg.norm(analytical - numerical)
        denom = np.linalg.norm(analytical) + np.linalg.norm(numerical)
        rel_error = diff / (denom + 1e-12)
        status = '✅ PASS' if rel_error < 1e-5 else '❌ FAIL'
        print(f"{name}: relative error = {rel_error:.2e}  {status}")

## Training Loop

In [ ]:
def train(X, y_onehot, hidden_size=64, lr=0.1, epochs=500):
    input_size  = X.shape[1]
    output_size = y_onehot.shape[1]
    W1, b1, W2, b2 = init_weights(input_size, hidden_size, output_size)

    losses = []
    for epoch in range(epochs):
        loss, cache = forward(X, W1, b1, W2, b2, y_onehot)
        grads = backward(cache, y_onehot, W2)

        W1 -= lr * grads['dW1']
        b1 -= lr * grads['db1']
        W2 -= lr * grads['dW2']
        b2 -= lr * grads['db2']

        losses.append(loss)
        if epoch % 100 == 0:
            preds = cache['a2'].argmax(axis=1)
            acc   = (preds == y_onehot.argmax(axis=1)).mean()
            print(f"epoch {epoch:4d}  loss: {loss:.4f}  acc: {acc:.2%}")

    return W1, b1, W2, b2, losses

## Run It

In [ ]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import OneHotEncoder

# toy dataset — 3 classes
X, y = make_classification(n_samples=300, n_features=10,
                            n_classes=3, n_informative=6,
                            random_state=42)
enc = OneHotEncoder(sparse_output=False)
y_onehot = enc.fit_transform(y.reshape(-1, 1))

print("Running gradient check on small network...")
W1, b1, W2, b2 = init_weights(10, 8, 3)
gradient_check(X[:5], y_onehot[:5], W1, b1, W2, b2)

In [ ]:
print("\nTraining...\n")
W1, b1, W2, b2, losses = train(X, y_onehot, hidden_size=64, lr=0.1, epochs=500)

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss — Backprop From Scratch')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Notes

Things I noticed working through this:
- The softmax + cross-entropy gradient being just `a2 - y` is genuinely elegant. The messy Jacobian of softmax cancels out cleanly.
- Numerical stability matters more than I expected — without the `max` subtraction in softmax, exp() overflows immediately.
- He initialisation vs random normal makes a visible difference to how fast it converges.

Next: implement this with momentum, then Adam — from scratch as well.